# Frozen Laya REAL / NOISY / NORMALIZED benchmark

This notebook performs the one-time official test evaluation after the Laya
checkpoint is ready. The **same frozen classifier** is used for all conditions:

`REAL = official test`, `NOISY = paired M1 synthetic corruption`,
`NORMALIZED = exactly NOISY passed through the selected TNT Hybrid V3 model`.

The execution switch is disabled by default to keep official test frozen until
all paths and manifests have been reviewed.

In [ ]:
from pathlib import Path
import json, sys

def find_repository_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "src/downstream").is_dir():
            return candidate
    raise RuntimeError("Open this notebook from inside the repository checkout")

REPO_ROOT = find_repository_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_DIR = REPO_ROOT / "data/external/injongo_intent_wol"
TEST_PATH = DATA_DIR / "test.jsonl"
LAYA_CHECKPOINT = REPO_ROOT / "checkpoints/downstream/laya_normalization_v1"
OUTPUT_DIR = REPO_ROOT / "results/downstream/laya_normalization_v1"
SEED = 2026
RUN_FROZEN_TEST = False  # review readiness below, then change once to True
DEVICE = None            # None selects CUDA when available
print("Laya checkpoint:", LAYA_CHECKPOINT)
print("Results:", OUTPUT_DIR)

## Readiness and locked TNT selection

The normalizer is not retrained. The repository's locked V3 selection is the
M1-pretrained checkpoint adapted on Gold train with learning rate 5e-5 and
`word_keep_threshold=0.50`, selected on Gold-dev WER. Hashes are checked before
inference. Its local checkpoint must be present because large weights are not
stored in Git.

In [ ]:
from src.downstream.tnt_normalizer import resolve_selected_tnt_checkpoint

assert LAYA_CHECKPOINT.is_dir(), "Run the fine-tuning notebook first."
training_manifest = json.loads((LAYA_CHECKPOINT / "training_manifest.json").read_text())
assert training_manifest["test_used"] is False
tnt_checkpoint, tnt_selection = resolve_selected_tnt_checkpoint(REPO_ROOT)
display({
    "laya_ready": True,
    "laya_best_epoch": training_manifest["best_epoch"],
    "tnt_checkpoint": str(tnt_checkpoint),
    "tnt_candidate": tnt_selection["candidate"],
    "tnt_selection_split": tnt_selection["selection_split"],
    "tnt_dev_metrics": tnt_selection["development_metrics"],
})

## One-time official test benchmark

Only this cell loads `test.jsonl`. It creates paired texts, normalizes NOISY in
batches, evaluates the same calibrated Laya checkpoint on all three conditions,
then writes metrics, per-class reports, confusion matrices, predictions and the
four-way paired analysis. No model is updated.

In [ ]:
from src.downstream.injongo import load_split
from src.downstream.benchmark import run_frozen_benchmark

if RUN_FROZEN_TEST:
    test_rows = load_split(TEST_PATH, "test")
    benchmark = run_frozen_benchmark(
        test_rows,
        laya_checkpoint=LAYA_CHECKPOINT,
        tnt_checkpoint=tnt_checkpoint,
        output_dir=OUTPUT_DIR,
        seed=SEED,
        device=DEVICE,
    )
    display(benchmark["metrics"])
else:
    print("Frozen test remains unopened. Set RUN_FROZEN_TEST=True when the pipeline is final.")

## Inspect saved results

This section is read-only and becomes available after the benchmark cell.
`recovery_rate` is emitted only when REAL outperforms NOISY, so a zero/negative
noise drop is never hidden by an unstable denominator.

In [ ]:
import pandas as pd

metrics_path = OUTPUT_DIR / "metrics.json"
if metrics_path.is_file():
    metrics = json.loads(metrics_path.read_text())
    display(pd.DataFrame({condition: values for condition, values in metrics.items() if condition in {"REAL", "NOISY", "NORMALIZED"}}).T)
    display(metrics["deltas"])
    display(metrics["paired"])
    display(pd.read_csv(OUTPUT_DIR / "paired_analysis.csv").head(20))
else:
    print("No benchmark results yet.")